# FoxTrack 03: Supervised models, leave one capture out

This notebook trains supervised models (logistic regression and gradient-boosted trees, Spark MLlib) on eight captures and tests each model on the ninth capture.

**Before you run it:** add the output of notebook 01 in **Input > Add Input > Notebook Output**, and set **Internet** to on.

**Why this design.** Each capture has one attack start. A model that trains and tests on the same capture cannot measure the delay at that start, because the start would be in the training data. Here the test capture is new to the model, so the result shows how well the model detects an attack type that it did not see.

**Method.**

1. Training data: a random sample of each capture, at most `N_PER_CLASS` benign and `N_PER_CLASS` malicious packets.
2. For each capture: train on the samples of the other eight captures, then score the test capture.
3. Threshold: from benign packets of the test capture (the same range as in notebook 02). The results are thus comparable with the baseline.

In [ ]:
# Settings.
REPO = "DiscM/DATA-226-FoxTrack"
BRANCH = "main"
OUT_DIR = "/kaggle/working/supervised"
N_PER_CLASS = 100_000                  # sample size for each class and capture
MODELS = {"logreg": {}, "gbt": {"maxIter": 50, "maxDepth": 5}}
TARGET_FPRS = (1e-2, 1e-3, 1e-4)
ALERT_RULES = ((1, 1), (3, 5), (10, 20))

In [ ]:
import shutil, subprocess, sys
if shutil.which("java") is None:
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless", shell=True, check=True)
!pip install -q pyspark mlflow

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None  # a public repository does not need a token
auth = f"{token}@" if token else ""
# A new clone on each run, so the code is always the latest version.
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)
done = subprocess.run(
    ["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
     f"https://{auth}github.com/{REPO}.git", "/kaggle/working/repo"],
    capture_output=True, text=True,
)
if done.returncode:
    raise RuntimeError("git clone failed. Examine REPO, BRANCH, and the GITHUB_TOKEN secret.")
sys.path.insert(0, "/kaggle/working/repo/src")
from foxtrack import etl, split, supervised
print("Code version:", subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Find the Parquet output of notebook 01.
from pathlib import Path
found = sorted(Path("/kaggle/input").rglob("summary.json")) + sorted(Path("/kaggle/working").glob("*/summary.json"))
if not found:
    raise FileNotFoundError("summary.json not found. Add the output of notebook 01 in Input > Add Input > Notebook Output.")
PARQUET_ROOT = found[0].parent
print("Parquet data:", PARQUET_ROOT)

## 1. Make the training sample

In [ ]:
%%time
import os
import mlflow
import pandas as pd

spark = etl.build_spark(driver_memory="20g")
summary = split.load_summary(PARQUET_ROOT)
os.makedirs(OUT_DIR, exist_ok=True)

sample_path = f"{OUT_DIR}/train_sample"
supervised.build_training_sample(spark, PARQUET_ROOT, summary, N_PER_CLASS).write.mode("overwrite").parquet(sample_path)
train = supervised.assemble(spark.read.parquet(sample_path)).cache()
train.groupBy("attack").pivot("label").count().orderBy("attack").show()

## 2. Train and test each fold

In [ ]:
%%time
import time
mlflow.set_tracking_uri("sqlite:////kaggle/working/mlflow.db")
mlflow.set_experiment("supervised_leave_one_out")
os.makedirs(f"{OUT_DIR}/scores", exist_ok=True)

all_results = []
for row in summary:
    s = split.onset_split(row)
    capture = spark.read.parquet(f"{PARQUET_ROOT}/features/attack={s.attack}")
    for name, params in MODELS.items():
        start = time.time()
        results, test = supervised.run_fold(train, capture, s, name, TARGET_FPRS, ALERT_RULES, **params)
        all_results.append(results)
        test.to_parquet(f"{OUT_DIR}/scores/{name}_{s.attack}.parquet", index=False)

        with mlflow.start_run(run_name=f"{name}_{s.attack}"):
            mlflow.log_params({"test_capture": s.attack, "model": name, "n_per_class": N_PER_CLASS, **params})
            mlflow.log_metric("roc_auc", results["roc_auc"].iloc[0])
            mlflow.log_metric("pr_auc", results["pr_auc"].iloc[0])
            for r in results.itertuples():
                tag = f"fpr{r.target_fpr:g}_k{r.k}n{r.n}"
                mlflow.log_metric(f"false_alarms_per_100k_{tag}", r.false_alarms_per_100k)
                mlflow.log_metric(f"packet_recall_{tag}", r.packet_recall)
                if r.detected:
                    mlflow.log_metric(f"delay_packets_{tag}", r.delay_packets)
        print(f"{s.attack} / {name}: done in {time.time() - start:.0f} s")

results = pd.concat(all_results, ignore_index=True)
results.to_csv(f"{OUT_DIR}/results.csv", index=False)

## 3. Results

**Ranking quality** on the test capture. These numbers do not use a threshold. A ROC-AUC near 0.5 shows that the model cannot separate the new attack from benign traffic.

In [ ]:
results.groupby(["attack", "model"])[["roc_auc", "pr_auc"]].first().unstack("model").round(3)

**Detection delay in packets** at a target false positive rate of 0.001 (empty = not detected). Columns are the model and the alert rule `k/n`.

In [ ]:
results["rule"] = results["k"].astype(str) + "/" + results["n"].astype(str)
at = results[results["target_fpr"] == 1e-3]
at.pivot(index="attack", columns=["model", "rule"], values="delay_packets")

**False alarms before the attack start**, for each 100,000 benign packets, at the same target rate.

In [ ]:
at.pivot(index="attack", columns=["model", "rule"], values="false_alarms_per_100k").round(1)

## 4. Keep the output

Select **Save Version > Save & Run All (Commit)**. The folder `supervised` then has `results.csv` and the test scores for each model and capture.

In [ ]:
print(results.drop(columns=["rule"]).to_csv(index=False))